In [ ]:
# オリジナルデータをコピー

import os
import shutil

def copy_directory_with_metadata(src, dst):
    """
    Copies all contents of the source directory to the destination directory,
    preserving metadata (e.g., timestamps).
    """
    if not os.path.exists(src):
        raise FileNotFoundError(f"Source directory '{src}' does not exist.")
    
    if not os.path.exists(dst):
        os.makedirs(dst)
    
    for root, dirs, files in os.walk(src):
        # Calculate the relative path from the source to current root
        relative_path = os.path.relpath(root, src)
        dst_root = os.path.join(dst, relative_path)
        
        # Create directories in the destination
        for dir_name in dirs:
            os.makedirs(os.path.join(dst_root, dir_name), exist_ok=True)
        
        # Copy files to the destination
        for file_name in files:
            src_file = os.path.join(root, file_name)
            dst_file = os.path.join(dst_root, file_name)
            shutil.copy2(src_file, dst_file)  # Use copy2 to preserve metadata

# Paths
src_dir = "../mydata/00_original"
dst_dir = "../mydata/00_copy"

# Perform the copy
copy_directory_with_metadata(src_dir, dst_dir)

In [ ]:
# 不要なファイルの削除

import os

def remove_unwanted_files(root_dir, allowed_extensions):
    """
    Removes files that do not have one of the allowed extensions from the specified root directory.
    
    Parameters:
        root_dir (str): The root directory to process.
        allowed_extensions (set): A set of allowed file extensions (e.g., {".jpg", ".png"}).
    """
    removed_files = []
    for root, _, files in os.walk(root_dir):
        for file_name in files:
            file_path = os.path.join(root, file_name)
            _, ext = os.path.splitext(file_name)
            if ext.lower() not in allowed_extensions:
                os.remove(file_path)
                removed_files.append(file_path)
    
    # Print summary
    print(f"Removed {len(removed_files)} files:")
    for file in removed_files:
        print(file)

# Root directory
root_dir = "../mydata/00_copy"

# Allowed file extensions (case-insensitive)
allowed_extensions = {
    ".jpg", ".jpeg", ".png", ".tif", ".tiff", ".gif", ".bmp", ".webp", ".ico", ".hdr", ".psd",
    ".doc", ".docx", ".ppt", ".pptx", ".xls", ".xlsx"
}

# Run the function
remove_unwanted_files(root_dir, allowed_extensions)


In [ ]:
# 空のディレクトリの削除

import os

def remove_empty_directories(root_dir):
    """
    Recursively removes empty directories from the specified root directory.
    
    Parameters:
        root_dir (str): The root directory to process.
    """
    removed_dirs = []
    
    for root, dirs, files in os.walk(root_dir, topdown=False):  # Start from the bottom of the tree
        for dir_name in dirs:
            dir_path = os.path.join(root, dir_name)
            # Check if the directory is empty
            if not os.listdir(dir_path):  # If the directory is empty
                os.rmdir(dir_path)
                removed_dirs.append(dir_path)
    
    # Print summary
    print(f"Removed {len(removed_dirs)} empty directories:")
    for dir_path in removed_dirs:
        print(dir_path)

# Root directory
root_dir = "../mydata/00_copy"

# Run the function
remove_empty_directories(root_dir)


In [ ]:
# ファイル名・フォルダ名の半角化
# ファイル名・フォルダ名からスペース除去

import os
import shutil
import unicodedata

def normalize_name(name):
    """
    Normalize a file or directory name by:
    - Converting full-width alphanumeric characters and symbols to half-width.
    - Removing both half-width and full-width spaces.
    """
    normalized = unicodedata.normalize("NFKC", name)
    normalized = normalized.replace(" ", "").replace("　", "")
    return normalized

def generate_unique_name(base_path, name):
    """
    Generate a unique name to prevent collisions by appending a counter (_1, _2, etc.).
    """
    base_name, ext = os.path.splitext(name)
    counter = 1
    unique_name = name
    while os.path.exists(os.path.join(base_path, unique_name)):
        unique_name = f"{base_name}_{counter}{ext}"
        counter += 1
    return unique_name

def rename_files_and_directories(root_dir):
    """
    Recursively renames all files and directories under the root directory.
    - Converts full-width alphanumeric characters and symbols to half-width.
    - Removes half-width and full-width spaces.
    - Ensures unique names by appending a counter if collisions occur.
    """
    for root, dirs, files in os.walk(root_dir, topdown=False):
        # Rename files
        for file_name in files:
            old_path = os.path.join(root, file_name)
            new_name = normalize_name(file_name)
            new_name = generate_unique_name(root, new_name)
            new_path = os.path.join(root, new_name)
            if old_path != new_path:
                shutil.move(old_path, new_path)
                print(f"Renamed file: {old_path} -> {new_path}")
        
        # Rename directories
        for dir_name in dirs:
            old_path = os.path.join(root, dir_name)
            new_name = normalize_name(dir_name)
            new_name = generate_unique_name(root, new_name)
            new_path = os.path.join(root, new_name)
            if old_path != new_path:
                shutil.move(old_path, new_path)
                print(f"Renamed directory: {old_path} -> {new_path}")

# Root directory
root_dir = "../mydata/00_copy"

# Run the function
rename_files_and_directories(root_dir)

In [ ]:
# 余計な"_1" を削除

import os
import re
import shutil

def remove_trailing_number_suffix(name):
    """
    Remove trailing '_<number>' from the name (if any) while preserving the extension for files.
    """
    # Separate base name and extension
    base_name, ext = os.path.splitext(name)
    # Regular expression to match '_<number>' at the end of the base name
    while True:
        new_name = re.sub(r"_(\d+)$", "", base_name)  # Remove trailing '_<number>'
        if new_name == base_name:  # If no change, stop processing
            break
        base_name = new_name
    return base_name + ext  # Reattach the extension for files

def generate_unique_name(base_path, name):
    """
    Ensure the name is unique within the directory. If a conflict occurs, return the original name.
    """
    new_path = os.path.join(base_path, name)
    if not os.path.exists(new_path):
        return name
    return None  # Return None to indicate a conflict

def process_directories_and_files(root_dir):
    """
    Process directories and files under the root directory:
    - Remove '_<number>' suffix from directory and file names.
    - Skip removal if it causes a name conflict.
    """
    for root, dirs, files in os.walk(root_dir, topdown=False):  # Process from bottom up
        # Process directories
        for dir_name in dirs:
            old_path = os.path.join(root, dir_name)
            new_name = remove_trailing_number_suffix(dir_name)
            if new_name != dir_name:
                unique_name = generate_unique_name(root, new_name)
                if unique_name:
                    new_path = os.path.join(root, unique_name)
                    shutil.move(old_path, new_path)
                    print(f"Renamed directory: {old_path} -> {new_path}")
                else:
                    print(f"Conflict detected, skipped renaming: {old_path}")
        
        # Process files
        for file_name in files:
            old_path = os.path.join(root, file_name)
            new_name = remove_trailing_number_suffix(file_name)
            if new_name != file_name:
                unique_name = generate_unique_name(root, new_name)
                if unique_name:
                    new_path = os.path.join(root, unique_name)
                    shutil.move(old_path, new_path)
                    print(f"Renamed file: {old_path} -> {new_path}")
                else:
                    print(f"Conflict detected, skipped renaming: {old_path}")

# Root directory
root_dir = "../mydata/00_copy"

# Run the function
process_directories_and_files(root_dir)

In [ ]:
# 単独ファイルの処理
# まずは目視でファイルを移動しておく

import os
import shutil

# 残したいファイルの拡張子
allowed_extensions = {
    ".jpg", ".jpeg", ".png", ".tif", ".tiff", ".gif", ".bmp", ".webp", ".ico", ".hdr", ".psd",
    ".doc", ".docx", ".ppt", ".pptx", ".xls", ".xlsx"
}

def organize_files_in_sogoshinryobu(root_dir):
    """
    Organize standalone files directly under "◯◯行" directories in "総合診療部".
    - Create a folder named after the file (excluding extension).
    - Move the file into the created folder.
    - Skip the process if a folder with the same name exists.
    """
    # Path to the "総合診療部" directory
    sogoshinryobu_dir = os.path.join(root_dir, "総合診療部")

    # Check if the directory exists
    if not os.path.exists(sogoshinryobu_dir):
        print(f"Error: The directory '{sogoshinryobu_dir}' does not exist.")
        return

    # Process each "◯◯行" directory directly under "総合診療部"
    for row_dir in os.listdir(sogoshinryobu_dir):
        row_path = os.path.join(sogoshinryobu_dir, row_dir)
        if not os.path.isdir(row_path):  # Skip non-directory items
            continue

        # Process files directly under the "◯◯行" directory
        for file_name in os.listdir(row_path):
            file_path = os.path.join(row_path, file_name)
            if os.path.isfile(file_path):
                # Get the file extension
                _, ext = os.path.splitext(file_name)

                # Process only allowed file extensions
                if ext.lower() in allowed_extensions:
                    # Create folder name based on file name (without extension)
                    base_name = os.path.splitext(file_name)[0]
                    new_folder_path = os.path.join(row_path, base_name)

                    # Check if the folder already exists
                    if os.path.exists(new_folder_path) and not os.path.isdir(new_folder_path):
                        print(f"Error: A non-directory with the name '{base_name}' exists at {row_path}. Skipping.")
                        continue

                    # Create the folder if it doesn't exist
                    if not os.path.exists(new_folder_path):
                        os.makedirs(new_folder_path)

                    # Define the new file path inside the folder
                    new_file_path = os.path.join(new_folder_path, file_name)

                    # Move the file into the folder
                    if not os.path.exists(new_file_path):
                        shutil.move(file_path, new_file_path)
                        print(f"Moved: {file_path} -> {new_file_path}")
                    else:
                        print(f"Error: File '{file_name}' already exists in '{new_folder_path}'. Skipping.")

# Root directory
root_dir = "../mydata/00_copy"

# Run the function
organize_files_in_sogoshinryobu(root_dir)
